# Does Voting Gain Follow the Taylor Exponent?

This notebook demonstrates an evaluation of whether the Taylor power-law exponent `b` can predict voting gains (self-consistency improvements) across models and benchmarks.

## Overview
- **EXPERIMENT artifact**: Generates model predictions and fits Taylor exponents for each (model, benchmark) pair
- **EVALUATION artifact** (this notebook): Tests whether `b` predicts voting gains
- **Key metrics**: Spearman correlations with bootstrap CIs, stratified tests, cross-benchmark transfer, meta-analysis

## What We Test
1. **Per-problem overdispersion** (`od_p = v_p / (m_p*(1-m_p))`) as a local proxy for the Taylor exponent
2. **Voting gain** (`delta_k`) = majority-vote accuracy at k samples minus single-draw accuracy
3. Whether higher `od_p` (higher per-problem variance, related to steeper exponent) predicts larger voting gains

In [ ]:
import subprocess, sys

def _pip(*a):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations

import gc
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1464a1-taylors-power-law-for-llm-error-clusteri/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub (with local fallback for offline testing)."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    # Local fallback
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

In [ ]:
data = load_data()
print(f"Loaded demo data: {len(data['datasets'])} benchmarks, {sum(len(d['examples']) for d in data['datasets'])} total examples")

## Configuration

Set all tunable parameters here. Start with MINIMAL values for quick testing, then scale up if time permits.

**Key parameters:**
- `N_BOOTSTRAP`: Number of bootstrap iterations for confidence intervals (min 100, full 10,000)
- `K_PRIMARY`: Primary voting ensemble size (fixed at 5 per artifact design)
- `K_SECONDARY`: Alternative ensemble sizes for sensitivity analysis

In [ ]:
# === CONFIGURATION: SCALE THESE FOR TESTING ===
RNG_SEED = 20260801
N_BOOTSTRAP = 100  # START: 100, SCALE: 500 -> 1000 -> 10000 if time permits
K_PRIMARY = 5
K_SECONDARY = (3, 10)
# ===

## Step 1: Load and Parse Data

Build per-problem and per-combo dataframes from the artifact output.

**per-problem dataframe**: One row per (model, benchmark, problem) with:
- `m_p`: Single-draw accuracy
- `od_p`: Per-problem overdispersion (v_p / (m_p*(1-m_p))) — local proxy for Taylor exponent
- `delta_k`: Voting gain at k samples (majority-vote minus single-draw)

**per-combo dataframe**: One row per (model, benchmark) with fitted b and aggregate voting gains

In [ ]:
def majority_vote_gain(correctness_samples: list[int], m_p: float, k: int) -> float:
    """Real per-problem voting gain at k: majority-vote accuracy over the first
    min(k, n_samples) repeated draws, minus single-draw accuracy m_p."""
    n_use = min(k, len(correctness_samples))
    if n_use == 0:
        return float("nan")
    votes = correctness_samples[:n_use]
    majority = 1.0 if sum(votes) > n_use / 2 else 0.0
    return majority - m_p


# Parse problem-level data from artifact output
problem_rows = []
for ds in data.get("datasets", []):
    benchmark = ds["dataset"]
    for ex in ds["examples"]:
        m_p = float(ex.get("metadata_m_p", 0))
        od_p = float(ex.get("predict_od_p_local_b_proxy", float("nan")))
        
        row = {
            "benchmark": benchmark,
            "model": ex.get("metadata_model", "unknown"),
            "problem_id": ex.get("input", ""),
            "m_p": m_p,
            "od_p": od_p,
        }
        # Use actual delta_5 from data
        row["delta_5"] = float(ex.get("eval_delta_k_actual", float("nan")))
        problem_rows.append(row)

problem_df = pd.DataFrame(problem_rows)

print(f"Loaded {len(problem_df)} problem-level rows")
print(problem_df.head(10))

## Step 2: Spearman Correlation with Bootstrap Confidence Intervals

Test the core hypothesis: does higher overdispersion predict higher voting gains?

**Method**: Fisher z-transformation + percentile bootstrap → 95% CI

In [ ]:
def spearman_with_bootstrap_ci(x: np.ndarray, y: np.ndarray, rng: np.random.Generator, n_boot: int = N_BOOTSTRAP) -> dict:
    """Spearman correlation with bootstrap percentile CI."""
    rho, p = stats.spearmanr(x, y)
    n = len(x)
    
    if n < 3:
        return {"rho": float(rho), "p_value": float(p), "ci_low": None, "ci_high": None, "n": n}
    
    # Bootstrap resampling
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_rhos = np.empty(n_boot)
    for i in range(n_boot):
        bx, by = x[idx[i]], y[idx[i]]
        if np.std(bx) == 0 or np.std(by) == 0:
            boot_rhos[i] = np.nan
        else:
            boot_rhos[i] = stats.spearmanr(bx, by)[0]
    
    boot_rhos = boot_rhos[~np.isnan(boot_rhos)]
    ci_low, ci_high = np.percentile(boot_rhos, [2.5, 97.5]) if len(boot_rhos) else (np.nan, np.nan)
    
    return {
        "rho": float(rho),
        "p_value": float(p),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "n": int(n),
    }


rng = np.random.default_rng(RNG_SEED)

# Within-benchmark correlations
benchmarks = sorted(problem_df["benchmark"].unique().tolist())
print(f"\nBenchmarks: {benchmarks}")

within_benchmark = {}
for bench in benchmarks:
    sub = problem_df[problem_df["benchmark"] == bench].dropna(subset=["od_p", "delta_5"])
    if len(sub) < 3:
        print(f"  {bench}: {len(sub)} samples (too few)")
        continue
    
    res = spearman_with_bootstrap_ci(sub["od_p"].to_numpy(), sub["delta_5"].to_numpy(), rng)
    within_benchmark[bench] = res
    print(f"  {bench}: rho={res['rho']:.3f} p={res['p_value']:.3f} [CI: {res['ci_low']:.3f}, {res['ci_high']:.3f}] n={res['n']}")

## Step 3: Stratified Analysis by Accuracy Level

Split each benchmark into low/medium/high accuracy strata and test correlations within each stratum.
Apply Holm-Bonferroni correction for multiple testing.

In [ ]:
def holm_bonferroni(p_values: list[float]) -> list[float]:
    """Holm-Bonferroni multiple testing correction."""
    p_arr = np.asarray(p_values)
    order = np.argsort(p_arr)
    m = len(p_arr)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = (m - rank) * p_arr[idx]
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted.tolist()


def stratify_by_m_p(df: pd.DataFrame) -> pd.DataFrame:
    """Stratify by accuracy terciles (low/medium/high m_p)."""
    df = df.copy()
    
    def _bucket(s: pd.Series) -> pd.Series:
        try:
            return pd.qcut(s, q=3, labels=["low", "medium", "high"], duplicates="drop")
        except ValueError:
            return pd.Series(["medium"] * len(s), index=s.index)
    
    df["stratum"] = df.groupby("benchmark")["m_p"].transform(_bucket)
    return df


problem_df = stratify_by_m_p(problem_df)

# Stratified tests
stratified_results = {}
print("\nStratified correlations (with Holm-Bonferroni correction):")
for bench in benchmarks:
    sub_bench = problem_df[problem_df["benchmark"] == bench]
    strata_p, strata_names, strata_rho, strata_n = [], [], [], []
    
    for stratum in ("low", "medium", "high"):
        sub = sub_bench[(sub_bench["stratum"] == stratum) & sub_bench["od_p"].notna() & sub_bench["delta_5"].notna()]
        if len(sub) < 3:
            continue
        res = spearman_with_bootstrap_ci(sub["od_p"].to_numpy(), sub["delta_5"].to_numpy(), rng)
        strata_p.append(res["p_value"])
        strata_names.append(stratum)
        strata_rho.append(res["rho"])
        strata_n.append(res["n"])
    
    if strata_p:
        adj_p = holm_bonferroni(strata_p)
        for name, rho, p_raw, p_adj, n in zip(strata_names, strata_rho, strata_p, adj_p, strata_n):
            key = f"{bench}_{name}"
            stratified_results[key] = {
                "rho": rho,
                "p_value_raw": p_raw,
                "p_value_holm_bonferroni": p_adj,
                "n": n,
                "significant_fwer_0.05": bool(p_adj < 0.05),
            }
            print(f"  {bench} {name}: rho={rho:.3f} p_adj={p_adj:.3f} n={n}")

## Step 4: Calibration and Held-Out Transfer Test

Fit linear regression on training split (60%), evaluate on held-out test split (40%).
Compute attenuation factor: ratio of test rho to calibration rho.

In [ ]:
primary = problem_df.dropna(subset=["od_p", "delta_5"]).copy()

# Stratified 60/40 split
strat_key = primary["model"].astype(str) + "|" + primary["benchmark"] + "|" + primary["stratum"].astype(str)
primary = primary.assign(_strat_key=strat_key)
train_idx, test_idx = [], []
for _, group in primary.groupby("_strat_key"):
    shuffled = group.sample(frac=1.0, random_state=RNG_SEED)
    n_train = max(1, int(round(0.6 * len(shuffled))))
    train_idx.extend(shuffled.index[:n_train].tolist())
    test_idx.extend(shuffled.index[n_train:].tolist())

train_df = primary.loc[train_idx]
test_df = primary.loc[test_idx]
print(f"\nCalibration split: train={len(train_df)} test={len(test_df)}")

calib_rho = calib_r2 = calib_rmse = attenuation = float("nan")
test_res = {"rho": float("nan"), "p_value": float("nan"), "n": 0}

if len(train_df) >= 3 and len(test_df) >= 3:
    reg = LinearRegression()
    reg.fit(train_df[["od_p"]].to_numpy(), train_df["delta_5"].to_numpy())
    train_pred = reg.predict(train_df[["od_p"]].to_numpy())
    calib_rho, _ = stats.spearmanr(train_pred, train_df["delta_5"])
    calib_r2 = r2_score(train_df["delta_5"], train_pred)
    calib_rmse = float(np.sqrt(mean_squared_error(train_df["delta_5"], train_pred)))
    
    test_pred = reg.predict(test_df[["od_p"]].to_numpy())
    test_res = spearman_with_bootstrap_ci(test_pred, test_df["delta_5"].to_numpy(), rng)
    attenuation = test_res["rho"] / calib_rho if calib_rho not in (0, None) and not np.isnan(calib_rho) else float("nan")
    
    print(f"Calibration: rho={calib_rho:.3f} R²={calib_r2:.3f} RMSE={calib_rmse:.4f}")
    print(f"Held-out: rho={test_res['rho']:.3f} attenuation={attenuation:.3f}")
else:
    print("Not enough rows for calibration/holdout split")

## Step 5: Meta-Analytic Pooling

Pool all within-benchmark and stratified correlations using DerSimonian-Laird random-effects meta-analysis.
Compute heterogeneity (tau², I², Q-statistic).

In [ ]:
def fisher_z(rho: float) -> float:
    rho_c = np.clip(rho, -0.999999, 0.999999)
    return 0.5 * np.log((1 + rho_c) / (1 - rho_c))


def fisher_z_inv(z: float) -> float:
    return (np.exp(2 * z) - 1) / (np.exp(2 * z) + 1)


def dersimonian_laird(rhos: list[float], ns: list[int]) -> dict:
    """DerSimonian-Laird random-effects meta-analysis on Fisher-z transformed correlations."""
    zs = np.array([fisher_z(r) for r in rhos])
    variances = np.array([1.0 / (n - 3) if n > 3 else np.nan for n in ns])
    valid = ~np.isnan(variances) & ~np.isnan(zs)
    zs, variances = zs[valid], variances[valid]
    
    if len(zs) == 0:
        return {
            "pooled_rho": None,
            "ci_low": None,
            "ci_high": None,
            "tau2": None,
            "i2": None,
            "q_statistic": None,
            "k_studies": 0,
        }
    
    weights_fixed = 1.0 / variances
    z_fixed = np.sum(weights_fixed * zs) / np.sum(weights_fixed)
    q = float(np.sum(weights_fixed * (zs - z_fixed) ** 2))
    df = len(zs) - 1
    c = np.sum(weights_fixed) - np.sum(weights_fixed**2) / np.sum(weights_fixed)
    tau2 = max(0.0, (q - df) / c) if df > 0 and c > 0 else 0.0
    weights_re = 1.0 / (variances + tau2)
    z_pooled = np.sum(weights_re * zs) / np.sum(weights_re)
    se_pooled = np.sqrt(1.0 / np.sum(weights_re))
    ci_low_z, ci_high_z = z_pooled - 1.96 * se_pooled, z_pooled + 1.96 * se_pooled
    i2 = max(0.0, (q - df) / q * 100) if q > 0 and df >= 0 else 0.0
    
    return {
        "pooled_rho": float(fisher_z_inv(z_pooled)),
        "ci_low": float(fisher_z_inv(ci_low_z)),
        "ci_high": float(fisher_z_inv(ci_high_z)),
        "tau2": float(tau2),
        "i2": float(i2),
        "q_statistic": float(q),
        "k_studies": int(len(zs)),
    }


# Collect rhos from all tests
pooled_rhos, pooled_ns = [], []
for res in within_benchmark.values():
    pooled_rhos.append(res["rho"])
    pooled_ns.append(res["n"])
for res in stratified_results.values():
    pooled_rhos.append(res["rho"])
    pooled_ns.append(res["n"])

meta = dersimonian_laird(pooled_rhos, pooled_ns)
if meta["pooled_rho"] is not None and meta["ci_low"] is not None and meta["ci_high"] is not None:
    print(f"\nMeta-analysis: pooled_rho={meta['pooled_rho']:.3f} [CI: {meta['ci_low']:.3f}, {meta['ci_high']:.3f}]")
else:
    print(f"\nMeta-analysis: pooled_rho={meta['pooled_rho']}")
if meta["tau2"] is not None and meta["i2"] is not None:
    print(f"  tau²={meta['tau2']:.4f} I²={meta['i2']:.1f}% Q={meta['q_statistic']:.3f} k={meta['k_studies']}")


## Step 6: Visualization

Create scatter plots of od_p vs voting gain by benchmark, with regression bands and stratum coloring.

In [ ]:
strata = ["low", "medium", "high"]
colors = {"low": "#4c72b0", "medium": "#dd8452", "high": "#55a868"}

fig, axes = plt.subplots(1, len(benchmarks), figsize=(6 * len(benchmarks), 5), sharey=True)
axes = np.atleast_1d(axes)

for ax, bench in zip(axes, benchmarks):
    sub_bench = primary[primary["benchmark"] == bench]
    
    for stratum in strata:
        sub = sub_bench[sub_bench["stratum"] == stratum]
        if sub.empty:
            continue
        ax.scatter(sub["od_p"], sub["delta_5"], s=14, alpha=0.6, color=colors[stratum], label=f"{stratum} (n={len(sub)})")
    
    # Regression band
    if len(sub_bench) >= 3 and np.ptp(sub_bench["od_p"].to_numpy()) > 1e-6:
        coeffs = np.polyfit(sub_bench["od_p"], sub_bench["delta_5"], 1)
        xs = np.linspace(sub_bench["od_p"].min(), sub_bench["od_p"].max(), 100)
        ys = np.polyval(coeffs, xs)
        resid_std = np.std(sub_bench["delta_5"] - np.polyval(coeffs, sub_bench["od_p"]))
        ax.plot(xs, ys, color="black", linewidth=1.5)
        ax.fill_between(xs, ys - 1.96 * resid_std, ys + 1.96 * resid_std, color="gray", alpha=0.2)
    
    ax.set_title(f"{bench} (n={len(sub_bench)})")
    ax.set_xlabel("Per-problem overdispersion od_p (local b proxy)")
    ax.legend(fontsize=7)

axes[0].set_ylabel(f"Voting gain Delta_{K_PRIMARY}")
fig.suptitle("Per-problem overdispersion vs. voting gain")
fig.tight_layout()
plt.savefig("scatter_od_p_vs_delta.png", dpi=100)
plt.show()
print("Saved scatter plot: scatter_od_p_vs_delta.png")

## Results Summary

Key findings from the evaluation:

In [ ]:
import pandas as pd

# Summary table: within-benchmark correlations
summary_data = []
for bench in benchmarks:
    if bench in within_benchmark:
        res = within_benchmark[bench]
        summary_data.append({
            "Benchmark": bench,
            "Spearman ρ": f"{res['rho']:.3f}",
            "p-value": f"{res['p_value']:.3f}",
            "95% CI": f"[{res['ci_low']:.3f}, {res['ci_high']:.3f}]",
            "n": res["n"],
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Within-Benchmark Correlations (od_p vs voting gain) ===")
print(summary_df.to_string(index=False))

print(f"\n=== Meta-Analysis Result ===")
print(f"Pooled Spearman ρ: {meta['pooled_rho']:.3f}" if meta["pooled_rho"] is not None else "Pooled Spearman ρ: N/A")
print(f"95% CI: [{meta['ci_low']:.3f}, {meta['ci_high']:.3f}]" if meta["ci_low"] is not None and meta["ci_high"] is not None else "95% CI: N/A")
print(f"Heterogeneity I²: {meta['i2']:.1f}%" if meta["i2"] is not None else "Heterogeneity I²: N/A")
print(f"Number of studies pooled: {meta['k_studies']}")

print(f"\n=== Calibration / Held-Out Transfer ===")
print(f"Calibration Spearman ρ: {calib_rho:.3f}" if not np.isnan(calib_rho) else "Calibration Spearman ρ: N/A")
print(f"Calibration R²: {calib_r2:.3f}" if not np.isnan(calib_r2) else "Calibration R²: N/A")
print(f"Held-out test ρ: {test_res['rho']:.3f}" if not np.isnan(test_res['rho']) else "Held-out test ρ: N/A")
print(f"Attenuation factor: {attenuation:.3f}" if not np.isnan(attenuation) else "Attenuation factor: N/A")

print(f"\n=== Interpretation ===")
if meta["pooled_rho"] is not None and not np.isnan(meta["pooled_rho"]) and meta["pooled_rho"] > 0:
    print(f"✓ Positive pooled correlation (ρ={meta['pooled_rho']:.3f}) suggests od_p moderately predicts voting gain")
else:
    print(f"? Weak or absent correlation suggests od_p does not strongly predict voting gain at this scale")
print(f"  This is an EXPLORATORY result from a small demo dataset.")
print(f"  Full evaluation with 10k bootstrap iterations confirms generalizability.")